# WGAN — Wasserstein GAN (2017)

_Papers · Generative Models_

**Replace the GAN's classifier-and-log-loss with a Lipschitz 'critic' that estimates the Earth-Mover distance, giving stable training and a loss curve that actually tracks sample quality.**

---

This notebook is a practice scaffold for the **WGAN — Wasserstein GAN (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — See why Wasserstein gives a gradient where JS does not

The paper's Example 1 ("parallel lines") shows the core motivation. When two distributions don't overlap, the **Jensen-Shannon** divergence is a flat `log 2` — its slope is zero, so a GAN gets no signal. The **Wasserstein** distance is just `|theta|`, which has a usable slope toward `theta=0`.

We confirm this by finite-differencing both distances at `theta=0.1`: Wasserstein has slope `+1`, JS has slope `0`.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import math

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Worked Example 1 ("Learning parallel lines"): W=|theta| is smooth; JS=log2 is flat.
def W_dist(theta):
    return abs(theta)                                         # Wasserstein (Eq.1 result)
def JS_dist(theta):
    return 0.0 if theta == 0 else math.log(2)                # Jensen-Shannon

for th in (0.5, 0.1, 0.0):
    print(f"theta={th:>4}:  W={W_dist(th):.4f}   JS={JS_dist(th):.4f}")

# Gradient signal at theta=0.1: W has slope sign(theta)=+1; JS is constant so slope 0.
eps = 1e-6
gW  = (W_dist(0.1 + eps) - W_dist(0.1 - eps)) / (2 * eps)
gJS = (JS_dist(0.1 + eps) - JS_dist(0.1 - eps)) / (2 * eps)
print(f"at theta=0.1:  dW/dtheta ~ {gW:+.1f}  (usable)   dJS/dtheta ~ {gJS:+.1f}  (no signal)")
assert abs(W_dist(0.5) - 0.5) < 1e-9 and abs(JS_dist(0.1) - math.log(2)) < 1e-9 and round(gW) == 1

### Step 2 — Define the generator and the critic

The generator is a plain MLP that maps noise to a 28x28 image — the WGAN idea lives in the critic and the loss, not in `G`.

The **critic** looks like a discriminator but its final layer has **no sigmoid**: it outputs an unbounded real-valued score, not a probability. That score is what estimates the Earth-Mover distance.

In [ ]:
NZ = 64

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(NZ, 256), nn.ReLU(True),
            nn.Linear(256, 512), nn.ReLU(True),
            nn.Linear(512, 28 * 28), nn.Tanh())          # Tanh -> pixels in [-1,1]
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512), nn.LeakyReLU(0.2, True),
            nn.Linear(512, 256), nn.LeakyReLU(0.2, True),
            nn.Linear(256, 1))                            # NO sigmoid: outputs an unbounded score
    def forward(self, x):
        return self.net(x).view(-1)

G = Generator().to(device)
C = Critic().to(device)

### Step 3 — Load MNIST scaled to [-1, 1]

We normalize the pixels to `[-1, 1]` so the real data matches the generator's `Tanh` output range. A small subset and a data loader keep this toy run fast on CPU.

In [ ]:
tfm = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(data, range(8000)),
                                     batch_size=64, shuffle=True, drop_last=True)

### Step 4 — Train with weight clipping (Algorithm 1)

WGAN trains the critic `n_critic=5` times per generator step to maximize `E[C(real)] - E[C(fake)]` (Eq. 3). After every critic update we **clip the weights** to `[-c, c]`, a crude way to keep the critic roughly 1-Lipschitz — which is what makes its score gap a valid Wasserstein estimate.

The generator then takes one step to maximize `E[C(fake)]`. Watch the printout: the critic loss (= negative Wasserstein estimate) trends *up toward 0* as samples improve, so the loss actually tracks quality — the paper's Fig. 3 property.

In [ ]:
# WGAN training (Algorithm 1): RMSProp, n_critic=5, clip c=0.01, lr=5e-5.
C_ITERS, CLIP, LR = 5, 0.01, 5e-5
optC = torch.optim.RMSprop(C.parameters(), lr=LR)
optG = torch.optim.RMSprop(G.parameters(), lr=LR)
fixed_z = torch.randn(64, NZ, device=device)
real_std = next(iter(loader))[0].std().item()
print(f"real data pixel std ~ {real_std:.3f}  (target as samples improve)")

it = iter(loader)
def next_real():
    global it
    try:
        x, _ = next(it)
    except StopIteration:
        it = iter(loader)
        x, _ = next(it)
    return x.to(device)

for step in range(1, 1201):
    # (a) train the CRITIC n_critic times: maximize E[C(real)] - E[C(fake)]  (Eq.3)
    for _ in range(C_ITERS):
        x = next_real()
        n = x.size(0)
        z = torch.randn(n, NZ, device=device)
        fake = G(z).detach()
        lossC = -(C(x).mean() - C(fake).mean())          # minimize negative gap = maximize the gap
        optC.zero_grad()
        lossC.backward()
        optC.step()
        for p in C.parameters():                          # WEIGHT CLIPPING -> (crude) 1-Lipschitz
            p.data.clamp_(-CLIP, CLIP)
    # (b) one GENERATOR step: maximize E[C(fake)] -> minimize -E[C(fake)]
    z = torch.randn(64, NZ, device=device)
    lossG = -C(G(z)).mean()
    optG.zero_grad()
    lossG.backward()
    optG.step()
    if step % 200 == 0:
        with torch.no_grad():
            w_est = (C(next_real()).mean() - C(G(fixed_z)).mean()).item()  # ~ Wasserstein estimate
            s = G(fixed_z)
        print(f"step {step:>4}  critic_loss {lossC.item():+.4f}  W_est {w_est:+.4f}  "
              f"sample std {s.std().item():.3f}")

### Step 5 — The ablation, explained

What happens if we remove the weight clipping (delete the `p.data.clamp_(-CLIP, CLIP)` line)? Without it the critic is no longer Lipschitz: `|C|` grows large, the real-minus-fake gap stops being a Wasserstein distance, and the loss diverges instead of settling. The visualization panel below runs exactly that comparison.

## Visualize the data & results

_As a WGAN trains on MNIST, does the critic loss (its Wasserstein-distance estimate) shrink toward zero as samples improve — and does removing the weight clipping break that?_

### Step 1 — Re-create the data, generator, and critic factories

This panel is self-contained, so it reloads MNIST and defines small factory functions for the generator and critic. The critic again ends without a sigmoid, producing a raw score — the architectures match the reference implementation above.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

# Reproduces the qualitative WGAN effect: with weight clipping the critic loss is a smooth
# Wasserstein estimate that shrinks toward 0; without it, the loss diverges.
torch.manual_seed(0)
NZ = 64
tfm = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(data, range(8000)),
                                     batch_size=64, shuffle=True, drop_last=True)

def make_G():
    return nn.Sequential(nn.Linear(NZ, 256), nn.ReLU(True),
                         nn.Linear(256, 512), nn.ReLU(True),
                         nn.Linear(512, 28 * 28), nn.Tanh())

def make_C():                                   # critic: NO final sigmoid -> raw score
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(28 * 28, 512), nn.LeakyReLU(0.2, True),
                         nn.Linear(512, 256), nn.LeakyReLU(0.2, True),
                         nn.Linear(256, 1))

### Step 2 — Run clipped vs. un-clipped and compare

The `run` function trains a fresh WGAN, optionally applying the weight-clipping step. We call it once with `clip=0.01` (the real WGAN) and once with `clip=None` (the ablation).

The clipped run's critic loss rises smoothly toward 0 — its Wasserstein estimate shrinks as samples improve. The un-clipped run's loss explodes to large negative values: with no Lipschitz bound the gap is no longer a distance, and training diverges.

In [ ]:
def run(clip, steps=1200):
    torch.manual_seed(0)
    G, C = make_G(), make_C()
    oC = torch.optim.RMSprop(C.parameters(), 5e-5)
    oG = torch.optim.RMSprop(G.parameters(), 5e-5)
    it = iter(loader)
    losses = []
    def real():
        nonlocal it
        try:
            x, _ = next(it)
        except StopIteration:
            it = iter(loader)
            x, _ = next(it)
        return x
    for t in range(steps + 1):
        for _ in range(5):                       # n_critic = 5
            x = real()
            n = x.size(0)
            fk = G(torch.randn(n, NZ)).view(-1, 1, 28, 28).detach()
            lC = -(C(x).mean() - C(fk).mean())
            oC.zero_grad()
            lC.backward()
            oC.step()
            if clip is not None:                 # the WGAN weight-clipping step
                for p in C.parameters():
                    p.data.clamp_(-clip, clip)
        z = torch.randn(64, NZ)
        (-C(G(z).view(-1, 1, 28, 28)).mean()).backward()
        oG.step()
        oG.zero_grad()
        if t % 100 == 0 or t in (50, 150):
            losses.append((t, lC.item()))
    return losses

clipped = run(0.01)
noclip  = run(None)
print("WGAN (clip 0.01):", [[t, round(v, 3)] for t, v in clipped])
print("no clipping     :", [[t, round(v, 1)] for t, v in noclip])
# Clipped: critic loss rises smoothly toward 0 (Wasserstein estimate shrinks).
# No clip: critic loss explodes to large negative values -- training diverges.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working WGAN on MNIST whose critic loss steadily decreases and whose
            samples improve. You remove the weight clipping (delete the
            p.data.clamp_(-c,c) line), changing nothing else, and retrain. What happens to the
            critic's outputs, its loss curve, and the samples &mdash; and which part of the WGAN math does this
            isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only the per-step weight-clipping; keep the critic architecture, the EM loss, $n_{\text{critic}}=5$, RMSProp, and the data identical. — _An honest ablation changes exactly one thing &mdash; the Lipschitz-enforcement mechanism &mdash; so any change in behavior is attributable to it._
- Retrain and watch the magnitude of the critic's outputs: with nothing bounding its slope, the critic drives real-minus-fake apart without limit, so $|f_w|$ grows large and the "loss" (negative gap) shoots off rather than settling near zero. — _Eq. 2's duality only equals the Wasserstein distance when $f$ is 1-Lipschitz. Remove the bound and the gap is no longer a distance &mdash; it is an unbounded number with no fixed point._
- Conclude that the weight clipping is what makes the loss a meaningful Wasserstein estimate; without it the curve is no longer interpretable and training is unstable. — _Clipping is the (crude) stand-in for the Lipschitz constraint; it is the load-bearing piece that connects the trainable critic back to the EM distance._

**Answer:** Without clipping the critic is unconstrained, so it pushes the real-vs-fake score gap apart
                 without bound: the critic's outputs and gradients grow large / explode, the loss stops
                 settling toward zero, and training becomes unstable instead of producing a clean downward curve.
                 Since only the clipping changed, this isolates the Lipschitz constraint (enforced by
                 weight clipping, Eq. 2&ndash;3): it is what makes the critic's gap a valid Wasserstein-distance
                 estimate. The CODEVIZ panel shows the clipped run's loss descending smoothly while the
                 no-clip run diverges. (This very weakness is what WGAN-GP later fixes with a gradient
                 penalty.)

</details>

**Problem 2.** Using the paper's Example 1 numbers, compare the gradient signal a GAN versus a WGAN sees when
            the generator's line sits at $\theta = 0.1$ and you want to push it to $\theta = 0$. Compute the
            relevant distance and its slope for each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- WGAN: $W(\mathbb{P}_0,\mathbb{P}_\theta) = |\theta|$, so at $\theta=0.1$, $W=0.1$ and $\frac{dW}{d\theta} = \mathrm{sign}(\theta) = +1$. — _The Wasserstein distance changes linearly with the gap, so its derivative is a constant non-zero number pointing toward $\theta=0$ &mdash; a usable gradient._
- GAN (JS): for any $\theta\ne0$, $\mathrm{JS}(\mathbb{P}_0,\mathbb{P}_\theta)=\log 2\approx0.693$, so at $\theta=0.1$ the value is $0.693$ and $\frac{d}{d\theta}\log 2 = 0$. — _JS is constant wherever the supports are disjoint, so its derivative is exactly zero &mdash; no signal telling the generator which way to move._
- Conclude the WGAN's loss yields slope $+1$ toward the target while the GAN's yields slope $0$. — _This is precisely why WGAN trains where the GAN stalls: a smooth distance gives a gradient even when the distributions do not overlap._

**Answer:** At $\theta=0.1$: WGAN sees $W=0.1$ with slope $\frac{dW}{d\theta}=+1$ &mdash; a clear
                 push toward $\theta=0$. GAN sees $\mathrm{JS}=\log 2\approx0.693$ with slope $0$ &mdash;
                 no gradient at all, since JS is flat for every non-zero gap. Same goes for KL ($+\infty$) and TV
                 ($1$). The smooth Wasserstein distance is the only one of the four that gives the generator a
                 usable learning signal here &mdash; the paper's core motivation.

</details>

**Problem 3.** Your critic's weights, after clipping with $c=0.01$, all lie in $[-0.01, 0.01]$. A teammate suggests
            "just use $c=1$ so the critic has more room." Explain &mdash; using Eq. 2 &mdash; why a large clip
            value breaks the method, and what the principled fix (a later paper) is.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall Eq. 2: the gap equals the Wasserstein distance only over 1-Lipschitz $f$. Clipping to $[-c,c]$ bounds the critic's slope, keeping it $K$-Lipschitz for a $K$ tied to $c$. — _The duality is exact only when $f$'s slope is bounded; clipping is the crude lever that bounds it._
- Argue that a large $c$ lets weights (and hence the slope $K$) grow large, so $f_w$ is no longer (approximately) 1-Lipschitz and the gap over-estimates / mismeasures the distance, destabilizing training. — _Beyond the Lipschitz regime the gap is not a Wasserstein distance, so the loss stops being a faithful, smooth signal._
- Name the principled fix: WGAN-GP (Gulrajani 2017) replaces clipping with a gradient penalty that directly pushes the critic's gradient norm toward 1. — _It enforces the Lipschitz condition smoothly instead of via a brittle hard box, removing the $c$ tuning problem the paper itself flags._

**Answer:** Eq. 2 only equals the Wasserstein distance when the critic is (1-)Lipschitz. Weight clipping to
                 $[-c,c]$ bounds the slope; a large $c$ lets the weights and slope grow, so $f_w$ leaves
                 the Lipschitz regime, the gap is no longer a faithful distance, and training destabilizes. (Too
                 small a $c$ instead starves capacity and vanishes gradients &mdash; hence $c=0.01$ is a
                 fragile sweet spot the paper openly criticizes.) The principled fix is WGAN-GP, which
                 swaps the hard clip for a gradient penalty driving the critic's gradient norm toward
                 $1$.

</details>